# E1 — Phân tích test chính thức ba seeds

Chạy tại máy cá nhân. Nguồn là phiên master session 1; không dùng pilot hoặc validation tuning làm test results. Mặc định dùng bảng đã kiểm chứng trong GitHub; bật FULL_AUDIT để tái kiểm tra ZIP riêng tư/dataset và bootstrap từ raw predictions. Không training/tuning/inference GPU.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/master_session1.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.master_session1 import ARCHIVE_NAME, ARCHIVE_SHA256, BUNDLE_HASH, analyze
TABLES = ROOT / "results/tables/master_session1"
FULL_AUDIT = False  # True needs the private original ZIP and matching dataset; never trains.
REBUILD_FIGURES = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_MASTER_ARCHIVE", str(ROOT / "artifacts/master_session1" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, resamples=10000)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["E1_success"] == 16 and audit["E2_success"] == 6 and audit["E7_gpu_state"] == "DEFERRED"
print("Source SHA256:", ARCHIVE_SHA256)
print("Mode:", "raw ZIP re-audited" if FULL_AUDIT else "versioned audited aggregate tables; not a fresh raw re-audit")


## Mean, sample SD và đầy đủ Recall/NDCG/MRR

Neural models có ba seeds; MostPopular chỉ một run xác định, SD không xác định. Bảng bao gồm full/warm và K=5/10/20 (MRR@10).


In [ ]:
summary = pd.read_csv(TABLES / "metric_summary.csv")
display(summary[summary.experiment == "E1"])
per_seed = pd.read_csv(TABLES / "per_seed_metrics.csv")
display(per_seed[per_seed.experiment == "E1"])


## Paired bootstrap

Lấy trung bình contributions qua seeds trong mỗi user trước khi lấy mẫu lại 22.363 users. 10.000 resamples, percentile 95%, seed 42; baseline dùng cố định. Multinomial trên nhóm vector trùng nhau giữ đúng phân phối empirical bootstrap. CI điều kiện trên checkpoint đã học, không bao trùm đầy đủ retraining uncertainty, chưa điều chỉnh đa so sánh.


In [ ]:
paired = pd.read_csv(TABLES / "paired_bootstrap.csv")
display(paired[paired.experiment == "E1"])


In [ ]:
from gen_fig_master_session1 import generate
if REBUILD_FIGURES:
    generate(ROOT)
display(Image(filename=str(ROOT / "results/figures/master_session1/e1_test_metrics.png")))
display(Image(filename=str(ROOT / "results/figures/master_session1/e1_recall_ndcg_at_k.png")))
display(Image(filename=str(ROOT / "results/figures/master_session1/e1_paired_bootstrap.png")))
display(Image(filename=str(ROOT / "results/figures/master_session1/e1_training_validation.png")))


## Báo cáo và trạng thái

Đọc [báo cáo tiếng Việt](../docs/MASTER_SESSION1_ANALYSIS_VI.md). Phạm vi thu gọn do tài nguyên GPU và yêu cầu môn học, không theo mô hình có test score cao. E1 complete, E2 partial, E3 pending/reuse available, E4/E5 deferred; TIGER/CRAMER theory only. Không tuyên bố các phase chưa chạy đã hoàn tất.
